# 🧱 Detección de Grietas en Imágenes (Concreto)

Proyecto limpio y ordenado para entrenar una CNN que clasifica imágenes como:

- **Positive (Positivo)** → con grieta
- **Negative (Negativo)** → sin grieta

Además incluye una **app con cámara** (Gradio) que permite seguir entrenando el modelo con fotos nuevas tomadas desde la webcam o subidas desde el disco.

## 📋 Cómo usar
1. Sube `archive_extracted.zip` (dataset con carpetas `Positive/` y `Negative/`) y `Prueba.zip` a `/content` (o móntalos desde Drive con la celda 2).
2. **Ejecuta las celdas en orden** (Menú: *Runtime → Run all*).
3. Recomendado: *Runtime → Change runtime type → **GPU***.
4. La última sección lanza la app con cámara para seguir entrenando.

## 🧩 Estructura
| Celda | Función |
|---|---|
| 1 | Imports y configuración |
| 2 | (Opcional) Montar Drive |
| 3 | Descomprimir dataset (sin prompts) |
| 4 | Verificar clases y ver ejemplos |
| 5 | Construir modelo (CNN + augmentation + dropout) |
| 6 | Entrenar con EarlyStopping y curvas |
| 7 | Evaluar / predecir en `Prueba.zip` |
| 8 | (Opcional) Estimar inclinación de imagen (Hough) |
| 9 | (Opcional) Copiar modelo a Drive |
| 10–12 | App con cámara: predicción + entrenamiento incremental |


In [ ]:
import os
import math
import numpy
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_HEIGHT = 128
IMG_WIDTH  = 128
BATCH_SIZE = 32
EPOCHS     = 20
SEED       = 123

DATA_ZIP   = '/content/archive_extracted.zip'
DATA_DIR   = '/content/archive_extracted'
TEST_ZIP   = '/content/Prueba.zip'
TEST_DIR   = '/content/datos_prueba'
MODEL_PATH = '/content/modelo_grietas.keras'

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile

def descomprimir(zip_path, destino):
    if not os.path.exists(zip_path):
        print(f"No se encontro: {zip_path}")
        return False
    os.makedirs(destino, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(destino)
    print(f"Descomprimido en: {destino}")
    for nombre in sorted(os.listdir(destino)):
        ruta = os.path.join(destino, nombre)
        if os.path.isdir(ruta):
            print(f"   - {nombre}: {len(os.listdir(ruta))} imágenes")
    return True

descomprimir(DATA_ZIP, DATA_DIR)

In [ ]:
class_names = sorted(os.listdir(DATA_DIR))
print("Clases detectadas (orden de etiquetas):", class_names)

ds_ejemplo = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=9, seed=SEED
)

plt.figure(figsize=(9, 9))
for imgs, labels in ds_ejemplo.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(imgs[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.show()

In [ ]:
def construir_modelo():
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.05),
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

modelo = construir_modelo()
modelo.summary()

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training",
    seed=SEED, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation",
    seed=SEED, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True)
]

history = modelo.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS, callbacks=callbacks)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entreno')
plt.plot(history.history['val_accuracy'], label='Validacion')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entreno')
plt.plot(history.history['val_loss'], label='Validacion')
plt.title('Perdida'); plt.legend()
plt.show()

modelo.save(MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")

In [ ]:
if os.path.exists(TEST_ZIP):
    descomprimir(TEST_ZIP, TEST_DIR)

if not os.path.isdir(TEST_DIR):
    print("No hay carpeta de prueba. Sube Prueba.zip o crea la carpeta manualmente.")
else:
    try:
        test_ds = tf.keras.utils.image_dataset_from_directory(
            TEST_DIR, image_size=(IMG_HEIGHT, IMG_WIDTH),
            batch_size=BATCH_SIZE, shuffle=False
        )
        loss, accuracy = modelo.evaluate(test_ds)
        print(f"Accuracy en prueba: {accuracy * 100:.2f}%")

        try:
            from sklearn.metrics import confusion_matrix
            y_true = np.concatenate([y for _, y in test_ds])
            y_pred = (modelo.predict(test_ds) > 0.5).astype(int).ravel()
            print("Matriz de confusion:")
            print(confusion_matrix(y_true, y_pred))
        except Exception as e:
            print("(Matriz de confusion omitida:", e, ")")

    except ValueError:
        test_ds = tf.keras.utils.image_dataset_from_directory(
            TEST_DIR, labels=None, image_size=(IMG_HEIGHT, IMG_WIDTH),
            batch_size=BATCH_SIZE, shuffle=False
        )
        preds = modelo.predict(test_ds)
        print("Resultados:")
        for ruta, p in zip(test_ds.file_paths, preds):
            nombre = os.path.basename(ruta)
            clase = "Positivo (Con Grieta)" if p[0] > 0.5 else "Negativo (Sin Grieta)"
            print(f"  {nombre} -> {clase} (certeza: {p[0]:.4f})")

In [ ]:
def estimar_inclinacion(image_path, umbral=100):
    if not os.path.exists(image_path):
        print(f"No se encontro: {image_path}")
        return
    img = cv2.imread(image_path)
    if img is None:
        print("Error: no se pudo cargar la imagen.")
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi/180, umbral)

    img_lines = img.copy()
    angulos = []
    if lines is not None:
        for r_theta in lines:
            r, theta = np.array(r_theta[0], dtype=np.float64)
            angulos.append(math.degrees(theta))
            a, b = math.cos(theta), math.sin(theta)
            x0, y0 = a * r, b * r
            x1, y1 = int(x0 + 1000 * (-b)), int(y0 + 1000 * (a))
            x2, y2 = int(x0 - 1000 * (-b)), int(y0 - 1000 * (a))
            cv2.line(img_lines, (x1, y1), (x2, y2), (0, 0, 255), 2)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.title("Original")
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.subplot(1, 2, 2); plt.title("Lineas (Hough)")
    plt.imshow(cv2.cvtColor(img_lines, cv2.COLOR_BGR2RGB))
    plt.show()

    if not angulos:
        print("No se detectaron lineas.")
        return
    verticales = [a for a in angulos if a < 20 or a > 160]
    if verticales:
        prom = np.mean(verticales)
        print(f"Desviacion respecto a la vertical: {min(prom, 180 - prom):.2f} grados")
    else:
        print("No se detectaron lineas verticales claras.")

In [ ]:
!cp /content/modelo_grietas.keras "/content/drive/MyDrive/"

In [ ]:
import gradio as gr
from datetime import datetime

CLASES = sorted([d for d in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, d))]) or ["Negative", "Positive"]
print("Clases de la app:", CLASES)

CONTADOR_NUEVAS = 0
ENTRENAR_CADA = 5

try:
    _ = modelo
except NameError:
    if os.path.exists(MODEL_PATH):
        modelo = tf.keras.models.load_model(MODEL_PATH)
        print("Modelo cargado desde:", MODEL_PATH)
    else:
        modelo = construir_modelo()
        print("No habia modelo guardado; se creo uno nuevo (entrena primero para buenos resultados).")

In [ ]:
def predecir(imagen):
    if imagen is None:
        return "Sube una imagen o usa la camara."
    img = cv2.resize(imagen, (IMG_WIDTH, IMG_HEIGHT)).astype("float32")
    prob = float(modelo.predict(np.expand_dims(img, axis=0), verbose=0)[0][0])
    idx = int(prob > 0.5)
    certeza = prob if idx == 1 else 1 - prob
    return f"{CLASES[idx]}  (certeza: {certeza:.2%})"

def reentrenar(epochs=3):
    cuentas = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in CLASES}
    print("Tamano del dataset:", cuentas)
    if min(cuentas.values()) < 2:
        print("Cada clase necesita al menos 2 imagenes para reentrenar.")
        return
    kw = dict(seed=SEED, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)
    if min(cuentas.values()) >= 10:
        tr = tf.keras.utils.image_dataset_from_directory(
            DATA_DIR, validation_split=0.2, subset="training", **kw)
        va = tf.keras.utils.image_dataset_from_directory(
            DATA_DIR, validation_split=0.2, subset="validation", **kw)
        tr = tr.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
        va = va.cache().prefetch(tf.data.AUTOTUNE)
        modelo.fit(tr, validation_data=va, epochs=epochs)
    else:
        ds = tf.keras.utils.image_dataset_from_directory(DATA_DIR, **kw)
        ds = ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
        modelo.fit(ds, epochs=epochs)
    modelo.save(MODEL_PATH)
    print("Modelo reentrenado y guardado en:", MODEL_PATH)

def guardar_y_entrenar(imagen, etiqueta, entrenar_ahora=False):
    global CONTADOR_NUEVAS
    if imagen is None:
        return "Toma o sube una imagen primero."
    carpeta = os.path.join(DATA_DIR, etiqueta)
    os.makedirs(carpeta, exist_ok=True)
    nombre = f"camara_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jpg"
    ruta = os.path.join(carpeta, nombre)
    cv2.imwrite(ruta, cv2.cvtColor(imagen, cv2.COLOR_RGB2BGR))
    CONTADOR_NUEVAS += 1
    if entrenar_ahora or CONTADOR_NUEVAS >= ENTRENAR_CADA:
        reentrenar()
        CONTADOR_NUEVAS = 0
        return f"Guardada '{nombre}' en {etiqueta}. Modelo reentrenado."
    faltan = ENTRENAR_CADA - CONTADOR_NUEVAS
    return (f"Guardada '{nombre}' en {etiqueta}. "
            f"Reentrenare en {faltan} imagen(es) mas (o pulsa 'Reentrenar ahora').")

In [ ]:
with gr.Blocks(title="Detector de Grietas") as demo:
    gr.Markdown(
        "# 🧱 Detector de Grietas - entrena con tu camara\n"
        "Toma fotos, corrige la etiqueta si el modelo se equivoca y el modelo mejora solo."
    )

    with gr.Tab("🔍 Predecir"):
        with gr.Row():
            img_in = gr.Image(sources=["webcam", "upload"], type="numpy",
                              label="Cámara o imagen subida")
            btn_pred = gr.Button("🔎 Predecir", variant="primary")
        res = gr.Textbox(label="Resultado")
        btn_pred.click(predecir, inputs=[img_in], outputs=[res])

    with gr.Tab("📷 Entrenar"):
        gr.Markdown(
            "Toma la foto, marca la **etiqueta correcta** "
            "(corrige si el modelo se equivocó) y guarda. "
            f"El modelo se reentrena automáticamente cada {ENTRENAR_CADA} imágenes."
        )
        img_tr = gr.Image(sources=["webcam", "upload"], type="numpy", label="Toma la foto")
        etq = gr.Radio(CLASES, label="Etiqueta correcta")
        with gr.Row():
            btn_guardar = gr.Button("💾 Guardar imagen")
            btn_reent = gr.Button("🔄 Reentrenar ahora")
        msg = gr.Textbox(label="Estado")
        btn_guardar.click(guardar_y_entrenar, inputs=[img_tr, etq], outputs=[msg])
        btn_reent.click(lambda i, e: guardar_y_entrenar(i, e, True),
                        inputs=[img_tr, etq], outputs=[msg])

demo.launch(share=True)

## 💡 Tips para mejorar el resultado
- Si la accuracy de validación se estanca, sube `EPOCHS` o usa **transfer learning** (p. ej. `tf.keras.applications.MobileNetV2` con `include_top=False`).
- Si las clases están desbalanceadas, usa `class_weight` en `modelo.fit(...)`.
- El modelo se guarda como `modelo_grietas.keras`; para recargarlo: `tf.keras.models.load_model(MODEL_PATH)`.
- **Entrenamiento incremental:** las fotos que guardes desde la app quedan en `Positive/` y `Negative/`, así que también participan en futuros entrenamientos completos. Ojo: reentrenar con pocas imágenes puede sobreajustar; por eso el umbral `ENTRENAR_CADA = 5`.
- ¿Quieres una API en vez de la app? Usa `server_api.py` (FastAPI) en la misma carpeta del proyecto.
